# Aula 4 — Notebook 3

## Autonomia controlada, API e empacotamento

Este notebook fecha a trilha técnica com uma pergunta central de engenharia: como transformar um workflow agêntico em uma aplicação utilizável, auditável e segura para operação no mundo real?

Nesta etapa, o foco deixa de ser apenas a qualidade da resposta do modelo e passa a incluir o comportamento do sistema como produto:

- quem pode decidir o quê;
- quais ações podem ser executadas sem intervenção humana;
- quais situações exigem bloqueio ou aprovação;
- como expor o workflow por CLI e API sem duplicar regras;
- como empacotar o serviço para execução reprodutível.

A ideia principal é simples: agentes podem propor, mas o sistema é governado por contratos, políticas e rotas explícitas. Em outras palavras, autonomia aqui é desenhada com limites claros, não com permissões irrestritas.

## 1. Níveis de autonomia

Um erro comum em sistemas com LLM é confundir sofisticação de raciocínio com autoridade operacional. Um agente pode gerar uma recomendação excelente e, ainda assim, não ter permissão para executá-la automaticamente.

Neste projeto, autonomia é tratada por níveis:

- nível de interpretação: o agente pode analisar contexto, hipóteses e alternativas;
- nível de proposta: o agente pode estruturar uma ação recomendada;
- nível de execução: o código aplica políticas e decide se a ação pode seguir;
- nível de governança: decisões de risco ficam sujeitas a revisão humana.

Essa divisão evita que decisões críticas dependam apenas de texto persuasivo do modelo. Em termos práticos:

- os agentes interpretam, revisam e propõem;
- as ferramentas são executadas por funções controladas;
- ações externas são simuladas neste ambiente didático;
- risco médio, risco alto ou irreversibilidade exigem aprovação;
- ações proibidas são bloqueadas mesmo quando o LLM recomenda execução.

Use as duas células de código seguintes para observar esse contraste: uma ação de baixo risco tende a passar na política, enquanto uma ação de alto risco muda a rota para revisão.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
INCIDENT_AGENT_DIR = PROJECT_ROOT / "target_project" / "incident_agent"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(INCIDENT_AGENT_DIR))

from shared.aula_4.validation import validate_action_policy

safe_action = {
    "action_name": "collect_additional_metrics",
    "risk_level": "low",
    "reversible": True,
    "parameters": {"window_minutes": 15},
    "justification": "Faltam evidências para confirmar a hipótese.",
    "requires_approval_recommendation": False,
}

validate_action_policy(safe_action)

{'is_allowed': True, 'requires_approval': False, 'errors': []}

In [2]:
high_risk_action = {
    "action_name": "rotate_credential",
    "risk_level": "high",
    "reversible": False,
    "parameters": {"credential_id": "masked"},
    "justification": "Existe suspeita de exposição de credencial.",
    "requires_approval_recommendation": True,
}

validate_action_policy(high_risk_action)

{'is_allowed': True, 'requires_approval': True, 'errors': []}

## 2. Human-in-the-loop como interrupção intencional

Encaminhar um caso para uma pessoa não deve ser visto automaticamente como falha do sistema. Em cenários reais, isso costuma ser um mecanismo de controle de risco e responsabilidade.

No workflow desta aula, a revisão humana é disparada por condições explícitas, por exemplo:

- a validação falha e o limite de tentativas foi atingido;
- a investigação declara incerteza relevante;
- a ação proposta é de médio ou alto risco;
- a ação não é reversível;
- a política determina bloqueio.

Perceba o padrão: não é o sentimento do operador nem uma decisão ad hoc. É uma regra verificável e reproduzível no estado do workflow.

O estado final `human_review_required` representa uma interrupção controlada. Em uma evolução de produto, esse estado pode virar um item de fila com auditoria, trilha de aprovação e retomada autenticada do processo. Essa arquitetura reduz decisões implícitas e facilita conformidade.

## 3. CLI e API como portas de entrada

A CLI e a API são interfaces diferentes para o mesmo núcleo de execução. O objetivo arquitetural é evitar duplicação de regra de negócio.

Ambas chamam:

```python
run_workflow(incident, settings, knowledge_dir)
```

Esse desenho traz ganhos importantes para manutenção e testes:

- uma única implementação de fluxo para evoluir;
- menor risco de divergência entre comportamentos da CLI e da API;
- facilidade para depurar via terminal antes de expor endpoint HTTP;
- melhor cobertura de testes de integração.

Para executar a CLI corretamente, primeiro navegue no terminal até a raiz do projeto `alura-skills-and-go-agentic-engineering`. Isso garante que o pacote `target_project` esteja disponível no caminho de importação.

### CLI

```bash
python -m target_project.incident_agent.app.cli investigate --incident-file target_project/incident_agent/data/incidents/incident_documented.json --knowledge-dir target_project/incident_agent/data/knowledge --env-file target_project/incident_agent/.env
```

### API

```bash
uvicorn target_project.incident_agent.app.api:app --reload
```

Depois, acesse `/docs` para explorar e testar a API interativamente.

Sugestão didática: execute o mesmo incidente pela CLI e pela API e compare os resultados. O comportamento esperado é equivalente, variando apenas a forma de entrada.

In [ ]:
# Exemplo de payload enviado à API
api_payload = {
    "incident_id": "INC-2001",
    "title": "Falha em processamento",
    "description": "O job noturno falhou após timeout na origem de dados.",
    "severity": "high",
    "source": "monitoring",
}
api_payload

## 4. Empacotamento com Docker

Docker é usado aqui como mecanismo de reprodutibilidade, não como selo automático de prontidão para produção.

O valor didático do contêiner é padronizar o ambiente de execução:

- versão do Python;
- instalação das dependências;
- diretório de trabalho;
- comando de inicialização;
- porta exposta.

Com isso, o mesmo artefato tende a se comportar de forma consistente em máquinas diferentes, reduzindo o clássico problema de ambiente local divergente.

Ao mesmo tempo, empacotar não resolve, por si só, responsabilidades operacionais mais amplas. Ainda ficam fora do escopo desta aula:

- gerenciamento corporativo de segredos;
- autenticação e autorização;
- políticas de rede;
- escalabilidade e balanceamento;
- alta disponibilidade;
- observabilidade distribuída;
- governança e retenção de dados.

Ao ler o Dockerfile na próxima célula, foque em entender como ele codifica decisões de ambiente e inicialização, não apenas em copiar comandos.

In [3]:
dockerfile_path = PROJECT_ROOT / "target_project/incident_agent/Dockerfile"
print(dockerfile_path.read_text(encoding="utf-8"))

FROM python:3.11-slim

WORKDIR /app

COPY requirements-aula4.txt ./requirements-aula4.txt
COPY shared ./shared
COPY target_project ./target_project

RUN pip install --no-cache-dir -r requirements-aula4.txt

EXPOSE 8000

CMD ["uvicorn", "target_project.incident_agent.app.api:app", "--host", "0.0.0.0", "--port", "8000"]



## 5. Teste ponta a ponta sugerido

Para concluir a formação com visão de engenharia, rode cenários que exercitem não só a qualidade da resposta, mas também os limites e garantias do sistema.

| Cenário | Comportamento esperado |
|---|---|
| Incidente bem documentado | investigação baseada em evidências e proposta coerente |
| Evidências insuficientes | baixa confiança, nova tentativa ou revisão humana |
| Saída inválida | correção por retry até o limite definido |
| Ação de médio risco | interrupção para aprovação |
| Ação proibida | bloqueio determinístico imediato |
| Falha de configuração | encerramento rápido com erro claro |

Checklist de análise após cada execução:

- o caminho seguido no grafo foi o esperado?
- as políticas foram respeitadas sem exceções manuais?
- o estado final reflete claramente o motivo da decisão?
- os eventos e métricas permitem explicar o que ocorreu?

O objetivo não é apenas obter uma resposta plausível, e sim verificar se o sistema permanece previsível quando confrontado com casos difíceis.

## 6. Exercício final — escolhendo a arquitetura

Compare três alternativas para o mesmo problema:

1. um único agente com todas as responsabilidades;
2. quatro agentes especializados, como no projeto;
3. dois agentes, por exemplo executor e revisor.

A decisão não deve ser guiada por preferência estética nem por quantidade de agentes. Ela deve ser argumentada com critérios observáveis:

- complexidade da tarefa;
- necessidade de revisão independente;
- custo e consumo de tokens;
- latência total;
- explicabilidade e auditabilidade;
- volume de execução;
- impacto operacional de erros.

Para aprofundar, remova um agente do grafo e adapte o restante do fluxo. Depois responda:

- quais contratos precisaram mudar?
- quais validações ficaram mais críticas?
- houve ganho de simplicidade com perda aceitável de qualidade?

Esse exercício mostra um ponto-chave da disciplina: framework facilita orquestração, mas não substitui decisões arquiteturais conscientes.

## Encerramento do curso

A progressão completa pode ser lida como uma evolução de maturidade de engenharia:

```text
Aula 1: prompt -> fluxo -> agente -> ferramenta
Aula 2: agente unico -> responsabilidades especializadas -> multiagentes
Aula 3: resposta -> contexto controlado -> avaliacao -> confiabilidade
Aula 4: prototipo -> aplicacao organizada -> rastreabilidade -> autonomia controlada
```

A mensagem final é prática: um sistema agêntico não está pronto quando responde bem em uma demonstração. Ele precisa ser:

- compreensível para quem opera e mantém;
- testável em cenários normais e adversos;
- rastreável por estado, eventos e métricas;
- limitado por políticas explícitas;
- evolutivo sem perder governança.

Se os alunos saírem desta trilha com essa lente de projeto, o objetivo do curso foi cumprido: usar LLM com responsabilidade de engenharia, e não apenas com entusiasmo de protótipo.